# 로컬 실행용 데이터 다운로드

Google Drive에 있는 Phase5_Sixpack_Cech 관련 파일을 확인하고,
필요한 파일을 ZIP으로 묶어 다운로드합니다.

### 다운로드 옵션
1. **입력 데이터 (ParamSweep)** — Pos_*.dat + Types_*.dat (풀 파이프라인용)
2. **계산 결과 (Sixpack_Cech)** — Sixpack_Cech_*.npz (평가만 할 때)
3. **둘 다**

## 1. 드라이브 마운트

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive/', force_remount=True)

In [ ]:
import os, glob, shutil
from pathlib import Path

BASE_DIR = '/content/drive/MyDrive/URP'
VECTOR_DIR = os.path.join(BASE_DIR, '1224_Vectors')
SIXPACK_CECH_DIR = os.path.join(VECTOR_DIR, 'Sixpack_Cech')

print(f'BASE_DIR: {BASE_DIR}')
print(f'  exists: {os.path.exists(BASE_DIR)}')
print(f'VECTOR_DIR: {VECTOR_DIR}')
print(f'  exists: {os.path.exists(VECTOR_DIR)}')
print(f'SIXPACK_CECH_DIR: {SIXPACK_CECH_DIR}')
print(f'  exists: {os.path.exists(SIXPACK_CECH_DIR)}')

## 2. 파일 분포 확인

In [ ]:
# ── 2-1. ParamSweep 입력 데이터 확인 ──────────────────────────────────
print('=' * 60)
print('📂 ParamSweep 입력 데이터 (Pos_*.dat, Types_*.dat)')
print('=' * 60)

sweep_dirs = sorted(glob.glob(os.path.join(BASE_DIR, 'ParamSweep_*_Output')))
print(f'\nParamSweep 폴더 수: {len(sweep_dirs)}')

total_pos = 0
total_types = 0
total_size_mb = 0
missing_folders = []
incomplete_folders = []

for i in range(1, 513):
    folder = os.path.join(BASE_DIR, f'ParamSweep_{i}_Output')
    if not os.path.exists(folder):
        missing_folders.append(i)
        continue
    pos_files = glob.glob(os.path.join(folder, 'Pos_*.dat'))
    types_files = glob.glob(os.path.join(folder, 'Types_*.dat'))
    total_pos += len(pos_files)
    total_types += len(types_files)
    for f in pos_files + types_files:
        total_size_mb += os.path.getsize(f) / (1024 * 1024)
    if len(pos_files) == 0 or len(types_files) == 0:
        incomplete_folders.append(i)

print(f'Pos_*.dat 파일 수: {total_pos}')
print(f'Types_*.dat 파일 수: {total_types}')
print(f'총 크기: {total_size_mb:.1f} MB')

if missing_folders:
    print(f'\n⚠️ 누락된 폴더 ({len(missing_folders)}개):')
    if len(missing_folders) <= 20:
        print(f'  {missing_folders}')
    else:
        print(f'  처음 20개: {missing_folders[:20]} ...')
else:
    print('\n✅ 1~512 폴더 모두 존재')

if incomplete_folders:
    print(f'\n⚠️ Pos/Types 파일 누락 폴더 ({len(incomplete_folders)}개): {incomplete_folders[:20]}')
else:
    print('✅ 모든 폴더에 Pos/Types 파일 존재')

In [ ]:
# ── 2-2. Sixpack_Cech 벡터 파일 확인 ─────────────────────────────────
print('=' * 60)
print('📂 Sixpack_Cech 벡터 파일 (*.npz)')
print('=' * 60)

if os.path.exists(SIXPACK_CECH_DIR):
    npz_files = sorted(glob.glob(os.path.join(SIXPACK_CECH_DIR, 'Sixpack_Cech_*.npz')))
    print(f'\nNPZ 파일 수: {len(npz_files)}')
    npz_size_mb = sum(os.path.getsize(f) for f in npz_files) / (1024 * 1024)
    print(f'총 크기: {npz_size_mb:.1f} MB')

    # 누락 확인
    found_indices = set()
    for f in npz_files:
        try:
            idx = int(os.path.basename(f).split('_')[-1].split('.')[0])
            found_indices.add(idx)
        except:
            pass
    expected = set(range(1, 513))
    missing_npz = sorted(expected - found_indices)
    if missing_npz:
        print(f'\n⚠️ 누락된 NPZ ({len(missing_npz)}개):')
        if len(missing_npz) <= 20:
            print(f'  {missing_npz}')
        else:
            print(f'  처음 20개: {missing_npz[:20]} ...')
    else:
        print('✅ 1~512 NPZ 파일 모두 존재')
else:
    print('\n❌ Sixpack_Cech 디렉토리가 존재하지 않습니다.')
    npz_files = []

In [ ]:
# ── 2-3. 1224_Vectors 디렉토리 내 다른 벡터 파일들도 확인 ──────────────
print('=' * 60)
print('📂 1224_Vectors 전체 구조')
print('=' * 60)

if os.path.exists(VECTOR_DIR):
    for item in sorted(os.listdir(VECTOR_DIR)):
        item_path = os.path.join(VECTOR_DIR, item)
        if os.path.isdir(item_path):
            files_in = os.listdir(item_path)
            size_mb = sum(os.path.getsize(os.path.join(item_path, f)) 
                         for f in files_in if os.path.isfile(os.path.join(item_path, f))) / (1024*1024)
            print(f'  📁 {item}/ — {len(files_in)} files, {size_mb:.1f} MB')
        else:
            size_mb = os.path.getsize(item_path) / (1024*1024)
            print(f'  📄 {item} — {size_mb:.1f} MB')
else:
    print('❌ 1224_Vectors 디렉토리 없음')

## 3. 다운로드 옵션 선택

아래 셀에서 `DOWNLOAD_MODE`를 설정하세요:
- `'vectors_only'` → Sixpack_Cech NPZ 파일만 (평가용, 작은 용량)
- `'input_only'` → ParamSweep Pos/Types 파일만 (풀 파이프라인용, 큰 용량)
- `'both'` → 둘 다

In [ ]:
# ⚠️ 여기서 다운로드 모드를 선택하세요
DOWNLOAD_MODE = 'vectors_only'  # 'vectors_only' | 'input_only' | 'both'

print(f'선택된 모드: {DOWNLOAD_MODE}')

## 4. ZIP 생성 & 다운로드

In [ ]:
import zipfile
import time

TMP_DIR = '/content/download_tmp'
os.makedirs(TMP_DIR, exist_ok=True)

def zip_sixpack_cech_vectors():
    """Sixpack_Cech NPZ 파일을 ZIP으로 묶기"""
    zip_path = os.path.join(TMP_DIR, 'Sixpack_Cech_Vectors.zip')
    npz_list = sorted(glob.glob(os.path.join(SIXPACK_CECH_DIR, 'Sixpack_Cech_*.npz')))
    
    if not npz_list:
        print('❌ NPZ 파일이 없습니다.')
        return None
    
    print(f'ZIP 생성 중: {len(npz_list)}개 NPZ 파일...')
    start = time.time()
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, f in enumerate(npz_list):
            arcname = os.path.join('Sixpack_Cech', os.path.basename(f))
            zf.write(f, arcname)
            if (i + 1) % 100 == 0:
                print(f'  {i+1}/{len(npz_list)} 완료...')
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'✅ 완료: {zip_path} ({size_mb:.1f} MB, {time.time()-start:.1f}초)')
    return zip_path


def zip_paramsweep_input():
    """ParamSweep 입력 데이터를 ZIP으로 묶기"""
    zip_path = os.path.join(TMP_DIR, 'ParamSweep_Input.zip')
    
    print(f'ZIP 생성 중: ParamSweep 입력 데이터...')
    print('⚠️ 파일이 많으면 시간이 오래 걸릴 수 있습니다.')
    start = time.time()
    
    file_count = 0
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for idx in range(1, 513):
            folder = os.path.join(BASE_DIR, f'ParamSweep_{idx}_Output')
            if not os.path.exists(folder):
                continue
            for dat_file in glob.glob(os.path.join(folder, '*.dat')):
                arcname = os.path.join(f'ParamSweep_{idx}_Output', os.path.basename(dat_file))
                zf.write(dat_file, arcname)
                file_count += 1
            if idx % 100 == 0:
                print(f'  폴더 {idx}/512 처리 완료...')
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'✅ 완료: {zip_path} ({file_count}개 파일, {size_mb:.1f} MB, {time.time()-start:.1f}초)')
    return zip_path


# ── 모드에 따라 ZIP 생성 ──────────────────────────────────────────────
zip_paths = []

if DOWNLOAD_MODE in ('vectors_only', 'both'):
    p = zip_sixpack_cech_vectors()
    if p: zip_paths.append(p)

if DOWNLOAD_MODE in ('input_only', 'both'):
    p = zip_paramsweep_input()
    if p: zip_paths.append(p)

print(f'\n📦 생성된 ZIP: {len(zip_paths)}개')
for p in zip_paths:
    print(f'  {p} ({os.path.getsize(p)/(1024*1024):.1f} MB)')

In [ ]:
# ── 다운로드 실행 ────────────────────────────────────────────────────
# Colab의 files.download()를 사용하여 브라우저로 다운로드합니다.
# ⚠️ 파일이 크면 시간이 오래 걸릴 수 있습니다.

for p in zip_paths:
    print(f'⬇️ 다운로드 시작: {os.path.basename(p)}')
    files.download(p)
    print(f'✅ 다운로드 완료: {os.path.basename(p)}')

In [ ]:
# ── 임시 파일 정리 (선택사항) ─────────────────────────────────────────
# shutil.rmtree(TMP_DIR)
# print('🧹 임시 파일 정리 완료')